# 01 TT-Conv：直接forwardとbackwardの一致

00では、Conv kernelをTT-matrixへ分解し、TT coresからdense kernelを再構成しました。

このNotebookでは、**dense kernelを再構成せず、入力patchとTT coresから直接Conv出力を計算する経路**を実装します。さらに、dense kernel再構成経路とTT coreのgradientが一致することを確認します。


## 1. このNotebookで確認すること

1. 入力画像から得たpatchをTT-matrix forwardへ渡せる行列として表現する。
2. TT coresから直接計算した出力をConv feature mapへ戻す。
3. exact TTの直接出力が元のdense Conv出力と一致することを確認する。
4. rank制限TTの直接出力が、同じcoresから再構成したkernelの出力と一致することを確認する。
5. 直接経路と再構成経路で、TT coreのgradientが一致することを確認する。

### このNotebookでは行わないこと

- Fashion-MNISTのdatasetや学習済みCNNは使わない。
- rank sweepやtensorization比較は行わない。
- fine-tuningや複数層圧縮は行わない。
- groupsが1以外の畳み込みは扱わない。
- MACsやlatencyの優位性は主張しない。


## 2. importと再現性

00で使用したTT-matrix APIを再利用します。Notebook内でTT-SVD、dense再構成、TT-matrix forwardを再実装しません。


In [43]:
from __future__ import annotations
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import math
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        project_root = candidate
        break
else:
    raise RuntimeError("src/nn_compressionが見つかりません。")

from nn_compression.compression import (
    dense_to_tt_matrix_cores,
    tt_linear_forward,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.utils import set_seed

SEED = 42
DTYPE = torch.float64
DEVICE = torch.device("cpu")
RTOL = 1e-9
ATOL = 1e-10
set_seed(SEED)

print("project root:", project_root)
print("torch:", torch.__version__)


project root: D:\dev\nn-compression-svd-dmrg
torch: 2.11.0+cu128


## 3. 00と同じ小さいConv2dを用意する

01単体で実行できるように、00と同じ設定、seed、tensorizationを再現します。ここは既習内容なので完成コードとします。


In [44]:
BATCH_SIZE = 2
IN_CHANNELS = 4
OUT_CHANNELS = 8
KERNEL_SIZE = (3, 3)
STRIDE = (1, 1)
PADDING = (1, 1)
DILATION = (1, 1)
GROUPS = 1
INPUT_HW = (7, 7)

OUT_MODES = (2, 2, 2)
IN_MODES = (3, 3, 4)
TRUNCATED_MAX_RANK = 2

dense_conv = nn.Conv2d(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    kernel_size=KERNEL_SIZE,
    stride=STRIDE,
    padding=PADDING,
    dilation=DILATION,
    groups=GROUPS,
    bias=True,
    dtype=DTYPE,
    device=DEVICE,
)
input_tensor = torch.randn(
    BATCH_SIZE, IN_CHANNELS, *INPUT_HW, dtype=DTYPE, device=DEVICE
)
reference_output = dense_conv(input_tensor)

output_height = (
    INPUT_HW[0] + 2 * PADDING[0]
    - DILATION[0] * (KERNEL_SIZE[0] - 1) - 1
) // STRIDE[0] + 1
output_width = (
    INPUT_HW[1] + 2 * PADDING[1]
    - DILATION[1] * (KERNEL_SIZE[1] - 1) - 1
) // STRIDE[1] + 1
num_output_positions = output_height * output_width

assert GROUPS == 1
assert tuple(reference_output.shape) == (
    BATCH_SIZE, OUT_CHANNELS, output_height, output_width
)
assert math.prod(OUT_MODES) == OUT_CHANNELS
assert math.prod(IN_MODES) == IN_CHANNELS * math.prod(KERNEL_SIZE)
print("input shape :", tuple(input_tensor.shape))
print("output shape:", tuple(reference_output.shape))


input shape : (2, 4, 7, 7)
output shape: (2, 8, 7, 7)


In [45]:
weight_matrix = dense_conv.weight.reshape(OUT_CHANNELS, -1)
exact_cores = dense_to_tt_matrix_cores(
    weight_matrix, OUT_MODES, IN_MODES, max_rank=None
)
truncated_cores = dense_to_tt_matrix_cores(
    weight_matrix, OUT_MODES, IN_MODES, max_rank=TRUNCATED_MAX_RANK
)

exact_kernel = tt_matrix_to_dense(
    exact_cores, OUT_MODES, IN_MODES
).reshape_as(dense_conv.weight)
truncated_kernel = tt_matrix_to_dense(
    truncated_cores, OUT_MODES, IN_MODES
).reshape_as(dense_conv.weight)

exact_reconstructed_output = F.conv2d(
    input_tensor, exact_kernel, dense_conv.bias,
    stride=STRIDE, padding=PADDING, dilation=DILATION, groups=GROUPS,
)
truncated_reconstructed_output = F.conv2d(
    input_tensor, truncated_kernel, dense_conv.bias,
    stride=STRIDE, padding=PADDING, dilation=DILATION, groups=GROUPS,
)

torch.testing.assert_close(
    exact_reconstructed_output, reference_output, rtol=RTOL, atol=ATOL
)
print("exact ranks   :", tt_matrix_ranks(exact_cores))
print("truncated ranks:", tt_matrix_ranks(truncated_cores))


exact ranks   : (1, 6, 8, 1)
truncated ranks: (1, 2, 2, 1)


## 4. Conv2dをpatchごとの線形写像として見る

1つの出力位置で参照される入力要素数を

$$
N = C_{\mathrm{in}} K_H K_W
$$

出力位置数を

$$
L = H_{\mathrm{out}} W_{\mathrm{out}}
$$

とします。入力patchを集めると、shapeは

$$
P \in \mathbb{R}^{B \times N \times L}
$$

になります。各patchを1つのsampleとみなした行列をTT-matrix forwardへ入力し、得られた出力を元のbatch・channel・空間位置へ戻します。

ここで重要なのは、要素数だけでなく、batchと空間位置の対応を保つことです。


In [46]:
unfolded_patches = F.unfold(
    input_tensor,
    kernel_size=KERNEL_SIZE,
    dilation=DILATION,
    padding=PADDING,
    stride=STRIDE,
)

patch_width = IN_CHANNELS * math.prod(KERNEL_SIZE)
assert tuple(unfolded_patches.shape) == (
    BATCH_SIZE, patch_width, num_output_positions
)

print("input_tensor:", tuple(input_tensor.shape))
print("unfolded patches:", tuple(unfolded_patches.shape))


input_tensor: (2, 4, 7, 7)
unfolded patches: (2, 36, 49)


## 5. TODO 1：patch行列を作る

### このセルで確認すること

`unfolded_patches`が持つbatch・patch要素・空間位置の対応を保ったまま、各patchを1行とする2階Tensorを作ります。

### 入力

- `unfolded_patches`：shape `(2, 36, 49)`

### 出力

- `patch_matrix: torch.Tensor`
- shape `(98, 36)`

### 期待する検証結果

最初の49行がbatch 0、次の49行がbatch 1に対応し、それぞれの行が同じ空間位置のpatch要素を保持します。


In [47]:
# patch_matrix = unfolded_patches.permute(0,2,1).reshape(98,36)
patch_matrix = unfolded_patches.permute(0, 2, 1).reshape(
    BATCH_SIZE * num_output_positions,
    patch_width,
)

In [48]:
if "patch_matrix" not in globals():
    print("TODO 1未実装: patch_matrix")
else:
    assert isinstance(patch_matrix, torch.Tensor)
    assert tuple(patch_matrix.shape) == (
        BATCH_SIZE * num_output_positions, patch_width
    )
    torch.testing.assert_close(patch_matrix[0], unfolded_patches[0, :, 0])
    torch.testing.assert_close(
        patch_matrix[num_output_positions - 1],
        unfolded_patches[0, :, num_output_positions - 1],
    )
    torch.testing.assert_close(
        patch_matrix[num_output_positions], unfolded_patches[1, :, 0]
    )
    print("TODO 1 checks: OK")


TODO 1 checks: OK


## 6. TODO 2：TT coresから直接Conv出力を作る

### このセルで確認すること

各patchにTT-matrixを作用させ、出力channelと空間位置を持つConv feature mapへ戻します。dense kernelは作りません。

### 入力

- `patch_matrix`
- `exact_cores`と`truncated_cores`
- `dense_conv.bias`
- 出力shapeを決める定数

### 出力

- `exact_direct_output: torch.Tensor`
- `truncated_direct_output: torch.Tensor`
- どちらもshape `(2, 8, 7, 7)`

### 期待する検証結果

- exact TTの直接出力は元のdense Conv出力と一致する。
- rank制限TTの直接出力は、同じcoresから再構成したkernelの出力と一致する。
- rank制限TTの出力は元のdense Conv出力と一致する必要はない。


In [49]:
exact_direct_output = tt_linear_forward(
    patch_matrix,
    exact_cores,
    OUT_MODES,
    IN_MODES,
    bias=dense_conv.bias,
)
# (B*H*W, C_out) → (B, C_out, H, W)
exact_direct_output = exact_direct_output.reshape(
    BATCH_SIZE, output_height, output_width, OUT_CHANNELS
).permute(0, 3, 1, 2)

truncated_direct_output = tt_linear_forward(
    patch_matrix,
    truncated_cores,
    OUT_MODES,
    IN_MODES,
    bias=dense_conv.bias,
)
truncated_direct_output = truncated_direct_output.reshape(
    BATCH_SIZE, output_height, output_width, OUT_CHANNELS
).permute(0, 3, 1, 2)


In [50]:
todo2_names = ("exact_direct_output", "truncated_direct_output")
todo2_missing = [name for name in todo2_names if name not in globals()]
if todo2_missing:
    print("TODO 2未実装:", todo2_missing)
else:
    expected_shape = tuple(reference_output.shape)
    assert tuple(exact_direct_output.shape) == expected_shape
    assert tuple(truncated_direct_output.shape) == expected_shape
    torch.testing.assert_close(
        exact_direct_output, reference_output, rtol=RTOL, atol=ATOL
    )
    torch.testing.assert_close(
        truncated_direct_output,
        truncated_reconstructed_output,
        rtol=RTOL,
        atol=ATOL,
    )

    comparison_records = [
        {
            "comparison": "exact direct vs dense Conv",
            "rmse": float(
                torch.sqrt(torch.mean((exact_direct_output - reference_output) ** 2))
                .detach()
            ),
        },
        {
            "comparison": "rank-limited direct vs reconstructed",
            "rmse": float(
                torch.sqrt(
                    torch.mean(
                        (truncated_direct_output - truncated_reconstructed_output) ** 2
                    )
                ).detach()
            ),
        },
        {
            "comparison": "rank-limited direct vs dense Conv",
            "rmse": float(
                torch.sqrt(
                    torch.mean((truncated_direct_output - reference_output) ** 2)
                ).detach()
            ),
        },
    ]
    df_forward_comparison = pd.DataFrame(comparison_records)
    display(df_forward_comparison)
    print("TODO 2 checks: OK")


,comparison,rmse
0,exact direct vs dense Conv,1.235322e-15
1,rank-limited direct vs reconstructed,7.787415e-17
2,rank-limited direct vs dense Conv,4.402539e-01


TODO 2 checks: OK


## 7. 直接経路と再構成経路のbackward

同じrank制限TT coresから、次の2経路を作ります。

$$
\text{direct path}:
\quad \{G^{(k)}\} \rightarrow \widetilde{Y}_{\mathrm{direct}} \rightarrow \mathcal{L}_{\mathrm{direct}}
$$

$$
\text{reconstructed path}:
\quad \{G^{(k)}\} \rightarrow \widetilde{W} \rightarrow \widetilde{Y}_{\mathrm{reconstructed}} \rightarrow \mathcal{L}_{\mathrm{reconstructed}}
$$

forwardが同じ写像なら、同じtargetとlossに対する各coreのgradientも数値誤差内で一致するはずです。


In [51]:
direct_cores = nn.ParameterList([
    nn.Parameter(core.detach().clone()) for core in truncated_cores
])
reconstructed_cores = nn.ParameterList([
    nn.Parameter(core.detach().clone()) for core in truncated_cores
])

set_seed(SEED + 1)
gradient_target = torch.randn_like(truncated_reconstructed_output)
fixed_bias = dense_conv.bias.detach()

assert all(core.grad is None for core in direct_cores)
assert all(core.grad is None for core in reconstructed_cores)


## 8. TODO 3：TT coreのgradientを比較する

### このセルで確認すること

`direct_cores`と`reconstructed_cores`から2つのforward経路を作り、同じtargetとの平均二乗誤差をそれぞれ逆伝播します。

### 入力

- `patch_matrix`
- `direct_cores`と`reconstructed_cores`
- `gradient_target`
- `fixed_bias`

### 出力

- `direct_learnable_output`と`reconstructed_learnable_output`
- `direct_loss`と`reconstructed_loss`
- 両方のcore列に保存されたgradient

### 期待する検証結果

- 2つの出力とlossが一致する。
- 対応する全coreのgradientが存在し、finiteである。
- 対応するcore gradientが数値誤差内で一致する。


In [52]:
direct_learnable_output = tt_linear_forward(
    patch_matrix,
    list(direct_cores),
    OUT_MODES,
    IN_MODES,
    bias=fixed_bias,
)
direct_learnable_output = direct_learnable_output.reshape(
    BATCH_SIZE, output_height, output_width, OUT_CHANNELS
).permute(0, 3, 1, 2)

reconstructed_weight_matrix = tt_matrix_to_dense(
    list(reconstructed_cores), OUT_MODES, IN_MODES
)
reconstructed_kernel = reconstructed_weight_matrix.reshape_as(dense_conv.weight)
reconstructed_learnable_output = F.conv2d(
    input_tensor,
    reconstructed_kernel,
    fixed_bias,
    stride=STRIDE,
    padding=PADDING,
    dilation=DILATION,
    groups=GROUPS,
)

direct_loss = F.mse_loss(direct_learnable_output, gradient_target)
reconstructed_loss = F.mse_loss(reconstructed_learnable_output, gradient_target)

direct_loss.backward()
reconstructed_loss.backward()


In [53]:
todo3_names = (
    "direct_learnable_output",
    "reconstructed_learnable_output",
    "direct_loss",
    "reconstructed_loss",
)
todo3_missing = [name for name in todo3_names if name not in globals()]
if todo3_missing:
    print("TODO 3未実装:", todo3_missing)
else:
    torch.testing.assert_close(
        direct_learnable_output,
        reconstructed_learnable_output,
        rtol=RTOL,
        atol=ATOL,
    )
    torch.testing.assert_close(direct_loss, reconstructed_loss, rtol=RTOL, atol=ATOL)

    gradient_records = []
    for core_index, (direct_core, reconstructed_core) in enumerate(
        zip(direct_cores, reconstructed_cores)
    ):
        assert direct_core.grad is not None
        assert reconstructed_core.grad is not None
        assert direct_core.grad.shape == direct_core.shape
        assert reconstructed_core.grad.shape == reconstructed_core.shape
        assert torch.isfinite(direct_core.grad).all()
        assert torch.isfinite(reconstructed_core.grad).all()
        torch.testing.assert_close(
            direct_core.grad, reconstructed_core.grad, rtol=RTOL, atol=ATOL
        )
        gradient_records.append(
            {
                "core_index": core_index,
                "core_shape": tuple(direct_core.shape),
                "direct_gradient_norm": float(
                    torch.linalg.vector_norm(direct_core.grad)
                ),
                "reconstructed_gradient_norm": float(
                    torch.linalg.vector_norm(reconstructed_core.grad)
                ),
                "max_abs_gradient_difference": float(
                    torch.max(torch.abs(direct_core.grad - reconstructed_core.grad))
                ),
            }
        )

    df_gradient_comparison = pd.DataFrame(gradient_records)
    display(df_gradient_comparison)
    print("TODO 3 checks: OK")


,core_index,core_shape,direct_gradient_norm,reconstructed_gradient_norm,max_abs_gradient_difference
0,0,"(1, 2, 3, 2)",0.117466,0.117466,9.714451e-17
1,1,"(2, 2, 3, 2)",0.118728,0.118728,4.857226e-17
2,2,"(2, 2, 4, 1)",0.220505,0.220505,6.938894e-17


TODO 3 checks: OK


## 9. 実装後の確認項目

- [ ] patch行列のshapeとbatch・空間位置の対応を確認した
- [ ] direct pathでdense kernelを再構成していない
- [ ] exact TTの直接出力が元のdense Conv出力と一致した
- [ ] rank制限TTの直接出力が同じTTの再構成経路と一致した
- [ ] 圧縮近似誤差と実装経路の一致誤差を区別した
- [ ] 2経路の全TT core gradientが一致した

次のNotebookでは、この直接TT-ConvをFashion-MNISTの学習済みCNNへ適用する準備に進みます。rank sweepやfine-tuningは、対象層とbaselineを確認してから別Notebookで扱います。
